# 08 — Clusterização de produtos

**Objetivo:** agrupar produtos por **comportamento operacional** (severidade, violação de OLA, MTTR,
regularidade, participação de ruído e instabilidade), e não apenas por tamanho.

**Algoritmo:** K-Means (não supervisionado) · **métrica:** coeficiente de silhueta.

O `k` não é escolhido no olho: o modelo é treinado de k = 2 a 7 e fica o de maior silhueta.

> Pré-requisito: executar antes o `02_engenharia_de_features.ipynb`.

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

PASTA = Path.cwd()
DADOS = PASTA / 'dados'            # saída do notebook 02, lida pelos notebooks de modelo
ARTEFATOS = PASTA / 'artefatos'    # modelos treinados por estes notebooks
DADOS.mkdir(exist_ok=True)
ARTEFATOS.mkdir(exist_ok=True)

SEMENTE = 42

base = pd.read_pickle(DADOS / 'base.pkl')
print('Base carregada:', f"{len(base['incidentes']):,} incidentes |",
      f"regime a partir de {base['inicio_regime']:%d/%m/%Y}")

## 1. Perfil operacional por produto

In [ ]:
painel = base['painel']
produtos = painel[painel['dimensao'] == 'produto']

perfil = produtos.groupby('chave').agg(
    volume_medio=('total_kpi', 'mean'),
    desvio=('total_kpi', 'std'),
    share_criticos=('criticos', 'sum'),
    total_kpi=('total_kpi', 'sum'),
    violacoes=('violacoes', 'sum'),
    mttr=('mttr_medio', 'mean'),
    dias_ativos=('total_kpi', lambda s: float((s > 0).mean())),
    ruido=('monitoramento', 'sum'),
    bruto=('total', 'sum'),
)
perfil['share_criticos'] = perfil['share_criticos'] / perfil['total_kpi'].replace(0, np.nan)
perfil['taxa_violacao'] = perfil['violacoes'] / perfil['total_kpi'].replace(0, np.nan)
perfil['share_ruido'] = perfil['ruido'] / perfil['bruto'].replace(0, np.nan)      # aberto por monitoramento
perfil['instabilidade'] = perfil['desvio'] / perfil['volume_medio'].replace(0, np.nan)

COLUNAS = ['volume_medio', 'share_criticos', 'taxa_violacao', 'mttr', 'dias_ativos', 'share_ruido', 'instabilidade']
X = perfil[COLUNAS].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f'Produtos: {len(X)}')
X.round(3)

## 2. Padronização e busca do k pela silhueta

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

K_MAXIMO = 7
escala = StandardScaler().fit(X)
Z = escala.transform(X)

avaliacao, melhor = [], None
for k in range(2, min(K_MAXIMO, len(X) - 1) + 1):
    modelo_k = KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit(Z)
    silhueta = float(silhouette_score(Z, modelo_k.labels_))
    avaliacao.append({'k': k, 'silhueta': silhueta, 'inercia': float(modelo_k.inertia_)})
    if melhor is None or silhueta > melhor['silhueta']:
        melhor = {'k': k, 'silhueta': silhueta, 'modelo': modelo_k}

busca = pd.DataFrame(avaliacao)
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.plot(busca['k'], busca['silhueta'], marker='o', color='#2563eb'); a.set_xlabel('k'); a.set_title('Silhueta')
b.plot(busca['k'], busca['inercia'], marker='o', color='#94a3b8'); b.set_xlabel('k'); b.set_title('Inércia (cotovelo)')
plt.tight_layout(); plt.show()
print(f"k escolhido: {melhor['k']} | silhueta: {melhor['silhueta']:.4f}")
busca.round(4)

## 3. Perfil de cada grupo

In [ ]:
modelo = melhor['modelo']
rotulos = pd.Series(modelo.labels_, index=perfil.index, name='cluster')
grupos = perfil.assign(cluster=rotulos).groupby('cluster')[COLUNAS].mean().round(3)
grupos.insert(0, 'produtos', rotulos.value_counts().sort_index())
display(grupos)

for cluster in sorted(rotulos.unique()):
    print(f'Grupo {cluster}: {", ".join(sorted(rotulos[rotulos == cluster].index))}')

In [ ]:
from sklearn.decomposition import PCA

pontos = PCA(n_components=2, random_state=SEMENTE).fit_transform(Z)
fig, eixo = plt.subplots(figsize=(7, 5))
for cluster in sorted(rotulos.unique()):
    mascara = (rotulos == cluster).to_numpy()
    eixo.scatter(pontos[mascara, 0], pontos[mascara, 1], s=60, label=f'grupo {cluster}')
for nome, (px, py) in zip(perfil.index, pontos):
    eixo.annotate(nome, (px, py), fontsize=7, alpha=0.7)
eixo.set_title('Produtos no plano das 2 componentes principais (só para visualização)')
eixo.legend(); plt.show()

## 4. Artefato

In [ ]:
import joblib

joblib.dump({'modelo': modelo, 'escala': escala, 'colunas': COLUNAS, 'rotulos': rotulos.to_dict()},
            ARTEFATOS / 'model_cluster.joblib', compress=3)
print('Artefato gravado em', ARTEFATOS / 'model_cluster.joblib')